# MTPL frequency: scratch work

Use the first section as a complete disposable data-to-model sandbox: load or sample source data, transform it, fit ordinary SuperGLM objects, and inspect predictions. Nothing there updates the governed handoff or publishes a model. Move accepted data work into `01_data_ingestion.ipynb` and accepted model choices into `02_model_training.ipynb`. The final optional section can open a published RAW candidate and export categorical collapses for notebook 02.

In [ ]:
DATABASE_MODE = "local"
RUNTIME_MODULE = None
EXPECTED_REMOTE_DATABASE = ""
ALLOW_REMOTE_WRITES = False
REFRESH_LOCAL_RAW = False
MODEL_NAME = "MTPL_FREQ"
MODEL_LABEL = "Motor frequency"
DEPLOYMENT_SLOT = "MTPL_FREQ_UAT"
SCRATCH_SAMPLE_ROWS = 50_000  # Set to None to use every source row.
SCRATCH_RANDOM_SEED = 42
GROUPING_SOURCE_PACKAGE_VERSION = None  # None selects the latest published RAW package.
REPLACE_GROUPING_ARTIFACT = False

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sklearn.metrics import mean_tweedie_deviance  # noqa: E402
from superglm import Categorical, Numeric, Spline, SuperGLM, Tweedie  # noqa: E402
from superglm.editor import EditorSession  # noqa: E402
from sqlalchemy import text  # noqa: E402

from pricing_pipeline.data.fremtpl import load_fremtpl_raw  # noqa: E402
from pricing_pipeline.infra.schema import schema_names_from_connectable  # noqa: E402
from pricing_pipeline.modeling.scratch_benchmark import (  # noqa: E402
    fit_boosted_blend,
    superglm_edf_table,
    unconstrained_superglm_features,
)
from pricing_pipeline.notebook import (  # noqa: E402
    connect,
    export_level_groupings,
    list_candidate_versions,
    load_registered_model,
    open_candidate,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/mtpl_frequency"
GROUPING_ARTIFACT_PATH = MODEL_DIR / ".local" / "routine_groupings.joblib"

In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)

## Sandbox 1: load or sample disposable source data

Replace this query with any source, join, filter, or sample you want to investigate. Nothing here writes the governed model-frame handoff.

In [ ]:
if pricing.mode == "local":
    load_fremtpl_raw(pricing.engine, replace=REFRESH_LOCAL_RAW)
schemas = schema_names_from_connectable(pricing.engine)
scratch_raw = pd.read_sql_query(
    text(f"SELECT * FROM {schemas.pricing}.FREMTPL_RAW ORDER BY IDpol"),
    pricing.engine,
)
if SCRATCH_SAMPLE_ROWS is not None and len(scratch_raw) > SCRATCH_SAMPLE_ROWS:
    scratch_raw = scratch_raw.sample(
        n=SCRATCH_SAMPLE_ROWS,
        random_state=SCRATCH_RANDOM_SEED,
    )
scratch_raw = scratch_raw.sort_values("IDpol").reset_index(drop=True)
display({"Rows": len(scratch_raw), "Columns": len(scratch_raw.columns)})
display(scratch_raw.head())

In [ ]:
# Blank ingestion area: replace or extend scratch_raw however you like.
# scratch_raw = pd.read_csv("...")
# scratch_raw = pd.read_sql_query("SELECT ...", pricing.engine)

## Sandbox 2: clean and engineer disposable features

Work on `scratch_frame` so the source sample stays easy to recover. Copy only accepted transformations into notebook 01.

In [ ]:
scratch_frame = scratch_raw.copy()
scratch_frame["CandidateLogDensity"] = np.log(
    scratch_frame["Density"].clip(lower=1.0)
)
scratch_frame["ScratchLogExposure"] = np.log(
    scratch_frame["Exposure"].clip(lower=1e-12)
)
display(scratch_frame.groupby("Area")["CandidateLogDensity"].describe())
display(scratch_frame.head())

In [ ]:
# Blank feature area: add plots, joins, filters, or alternative columns.
# scratch_frame["another_candidate"] = ...

## Sandbox 3: define and fit a disposable model

Use ordinary SuperGLM objects here. This model exists only in memory: it is not registered, manifested, built, staged, or published. Copy accepted feature and model choices into notebook 02.

In [ ]:
SCRATCH_FAMILY = "poisson"  # For compound Tweedie: Tweedie(p=1.6)
SCRATCH_TARGET = "ClaimNb"
SCRATCH_OFFSET_COLUMN = "ScratchLogExposure"
SCRATCH_FEATURES = {
    "VehAge": Spline(),
    "DrivAge": Spline(),
    "BonusMalus": Spline(),
    "CandidateLogDensity": Numeric(),
    "Area": Categorical(),
    "VehPower": Categorical(),
    "VehBrand": Categorical(),
    "VehGas": Categorical(),
    "Region": Categorical(),
}
scratch_X = scratch_frame.loc[:, list(SCRATCH_FEATURES)]
scratch_y = scratch_frame[SCRATCH_TARGET].astype(float)
scratch_offset = scratch_frame[SCRATCH_OFFSET_COLUMN].astype(float)

In [ ]:
scratch_model = SuperGLM(
    family=SCRATCH_FAMILY,
    selection_penalty=0.0,
    discrete=True,
    n_bins=256,
    features=SCRATCH_FEATURES,
).fit(scratch_X, scratch_y, offset=scratch_offset)

## Sandbox 4: inspect, compare, and iterate

In [ ]:
scratch_predictions = scratch_model.predict(scratch_X, offset=scratch_offset)
scratch_results = pd.DataFrame({
    "actual": scratch_y,
    "prediction": scratch_predictions,
})
display({
    "Rows fitted": len(scratch_results),
    "Actual mean": float(scratch_results["actual"].mean()),
    "Predicted mean": float(scratch_results["prediction"].mean()),
    "Mean absolute error": float(
        np.mean(
            np.abs(
                scratch_results["actual"]
                - scratch_results["prediction"]
            )
        )
    ),
})
display(scratch_results.head())

In [ ]:
# Blank modelling area: try another feature map, family, penalty, or split.
# alternative_features = {...}
# alternative_model = SuperGLM(...).fit(...)

## Optional: fully unconstrained SuperGLM benchmark

This is a decision-light technical benchmark: raw categorical levels, no groupings, no monotonic or shape constraints, and data-driven spline knots and REML lambdas. The reference level and spline penalty remain necessary model mechanics. It stays in memory and cannot be published or deployed from this notebook.

In [ ]:
UNCONSTRAINED_CATEGORICAL_COLUMNS = (
    "Area",
    "VehPower",
    "VehBrand",
    "VehGas",
    "Region",
)
UNCONSTRAINED_ORDERED_COLUMNS = {}
UNCONSTRAINED_LINEAR_COLUMNS = ()

unconstrained_features = unconstrained_superglm_features(
    scratch_X,
    categorical_columns=UNCONSTRAINED_CATEGORICAL_COLUMNS,
    ordered_columns=UNCONSTRAINED_ORDERED_COLUMNS,
    linear_columns=UNCONSTRAINED_LINEAR_COLUMNS,
    spline_kind="ps",
    k=10,
    knot_strategy="quantile_tempered",
    knot_alpha=0.2,
)
unconstrained_model = SuperGLM(
    family=SCRATCH_FAMILY,
    selection_penalty=0.0,
    discrete=True,
    n_bins=256,
    features=unconstrained_features,
).fit_reml(
    scratch_X,
    scratch_y,
    offset=scratch_offset,
    runtime_validation="skip",
)
unconstrained_predictions = unconstrained_model.predict(
    scratch_X,
    offset=scratch_offset,
)
display({
    "Rows fitted": len(scratch_y),
    "Mean unit deviance": float(
        mean_tweedie_deviance(
            scratch_y,
            np.clip(unconstrained_predictions, 1e-12, None),
            power=float(getattr(unconstrained_model.family, "p", 1.0)),
        )
    ),
    "Evidence": "in-sample technical fit; use OOF evidence below for comparison",
})
display(superglm_edf_table(unconstrained_model))

## Optional: CatBoost + LightGBM + XGBoost blend

Install once with `uv sync --extra scratch`, restart the kernel, then run this cell. Each learner is fitted out-of-fold and non-negative convex blend weights are learned only from those held-out predictions. Exposure is handled as an exposure-weighted rate target, then converted back to expected claims. These weights are a technical predictive benchmark, not a proposed pricing blend. A governed GAM/GBM blend should impose its chosen GAM floor and be assessed through tail, calibration, and stability evidence. This remains scratch-only: it does not touch SQL or the candidate lane.

In [ ]:
boosted_blend = fit_boosted_blend(
    scratch_X,
    scratch_y,
    categorical_columns=UNCONSTRAINED_CATEGORICAL_COLUMNS,
    exposure=scratch_frame.loc[scratch_X.index, "Exposure"],
    n_splits=5,
    random_state=SCRATCH_RANDOM_SEED,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    reference_superglm=unconstrained_model,
)
display(boosted_blend.metrics.sort_values("mean_unit_deviance"))
display(pd.Series(boosted_blend.weights, name="OOF convex weight"))
display(boosted_blend.oof_predictions.head())

## Optional: create the routine level-grouping artifact

This section is read-only in SQL but requires `DATABASE_MODE = "remote"`, because editable candidate bundles come from the remote workbench. Select levels in the widget and use **Collapse and refit** repeatedly across any categorical features you need.

In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
versions = list_candidate_versions(pricing, model=model)
raw_versions = versions.loc[versions["Kind"].eq("RAW")].copy()
display(raw_versions)
if raw_versions.empty:
    raise LookupError("No published RAW candidate is available for grouping.")
selected_package_version = (
    int(raw_versions.iloc[0]["Package"])
    if GROUPING_SOURCE_PACKAGE_VERSION is None
    else int(GROUPING_SOURCE_PACKAGE_VERSION)
)
if selected_package_version not in set(raw_versions["Package"].astype(int)):
    raise ValueError(
        "GROUPING_SOURCE_PACKAGE_VERSION is not in the displayed RAW list."
    )
grouping_candidate = open_candidate(
    pricing,
    model=model,
    package_version=selected_package_version,
)

In [ ]:
grouping_session = EditorSession.from_model(
    grouping_candidate.bundle.fitted_model,
    train_data=(
        grouping_candidate.bundle.X,
        grouping_candidate.bundle.y,
        grouping_candidate.bundle.sample_weight,
        grouping_candidate.bundle.offset,
    ),
    cv_report=grouping_candidate.bundle.cv_report,
)
display(grouping_session.widget())

## Export all current groupings

Run this after every intended collapse/refit has completed. The binary contains the real Python objects; the generated JSON sidecar is integrity and lineage evidence, not an analyst-edited configuration.

In [ ]:
grouping_artifact = export_level_groupings(
    grouping_candidate,
    editor_session=grouping_session,
    path=GROUPING_ARTIFACT_PATH,
    replace=REPLACE_GROUPING_ARTIFACT,
)
display(grouping_artifact)